<a href="https://colab.research.google.com/github/alextung2/ECE_285_Final_Project/blob/main/ece285_Medusa_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import torch
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("GPU 可用:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU 型号:", gpu.name)
    print(f"GPU 总显存: {gpu.total_memory / 1024**3:.1f} GiB")

for package in ["transformers", "accelerate", "bitsandbytes"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: 未安装")

Python: 3.13.16
PyTorch: 2.11.0+cu130
PyTorch CUDA: 13.0
GPU 可用: True
GPU 型号: Tesla T4
GPU 总显存: 14.6 GiB
transformers: 5.20.2
accelerate: 1.15.0
bitsandbytes: 未安装


In [2]:
!nvidia-smi

Wed Oct 28 18:29:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%pip install -q uv

In [4]:
import subprocess

subprocess.run(
    ["uv", "venv", "--python", "3.11", "/content/medusa-env"],
    check=True,
)

subprocess.run(
    [
        "uv", "pip", "install",
        "--python", "/content/medusa-env/bin/python",
        "torch==2.2.2",
        "--index-url", "https://download.pytorch.org/whl/cu121",
    ],
    check=True,
)

CompletedProcess(args=['uv', 'pip', 'install', '--python', '/content/medusa-env/bin/python', 'torch==2.2.2', '--index-url', 'https://download.pytorch.org/whl/cu121'], returncode=0)

In [5]:
from pathlib import Path
import subprocess

repo = Path("/content/Medusa")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/FasterDecoding/Medusa.git",
            str(repo),
        ],
        check=True,
    )

subprocess.run(
    [
        "uv", "pip", "install",
        "--python", "/content/medusa-env/bin/python",
        "torch==2.2.2",
        "numpy==1.26.4",
        "transformers==4.36.2",
        "accelerate==0.27.2",
        "huggingface-hub==0.25.2",
        "bitsandbytes==0.43.3",
        "fschat==0.2.36",
        "-e", str(repo),
    ],
    check=True,
)

CompletedProcess(args=['uv', 'pip', 'install', '--python', '/content/medusa-env/bin/python', 'torch==2.2.2', 'numpy==1.26.4', 'transformers==4.36.2', 'accelerate==0.27.2', 'huggingface-hub==0.25.2', 'bitsandbytes==0.43.3', 'fschat==0.2.36', '-e', '/content/Medusa'], returncode=0)

In [8]:
import subprocess

check_code = r"""
import sys
import importlib.util
import subprocess

# 仅在缺少 setuptools 时安装
if importlib.util.find_spec("setuptools") is None:
    subprocess.run(
        ["uv", "pip", "install", "--python", sys.executable, "setuptools"],
        check=True,
    )

import torch
import transformers
import bitsandbytes as bnb

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)

assert torch.cuda.is_available(), "未检测到 GPU"
print("GPU:", torch.cuda.get_device_name(0))

from medusa.model.medusa_model import MedusaModel
print("Medusa 导入成功")

with torch.inference_mode():
    layer = bnb.nn.Linear4bit(
        256, 256,
        bias=False,
        compute_dtype=torch.float16,
        quant_type="nf4",
    ).to("cuda")

    x = torch.randn(1, 256, device="cuda", dtype=torch.float16)
    y = layer(x)
    torch.cuda.synchronize()
    assert torch.isfinite(y).all().item(), "4-bit 运算出现无效数值"

print("4-bit GPU 运算成功")
print("环境检查通过")
"""

result = subprocess.run(
    ["/content/medusa-env/bin/python", "-u", "-c", check_code],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print("检查失败，具体错误：")
    print(result.stderr)
else:
    print("下一步可以开始加载模型。")

Python: 3.11.18
PyTorch: 2.2.2+cu121
Transformers: 4.36.2
bitsandbytes: 0.43.3
GPU: Tesla T4
Medusa 导入成功
4-bit GPU 运算成功
环境检查通过

下一步可以开始加载模型。


In [9]:
import subprocess

load_code = r"""
import torch
from transformers import BitsAndBytesConfig
from medusa.model.medusa_model import MedusaModel

model_id = "FasterDecoding/medusa-vicuna-7b-v1.3"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model. The first run requires downloading the weights...", flush=True)

model = MedusaModel.from_pretrained(
    model_id,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map={"": 0},
)

model.eval()

print("\nModel loaded successfully")
print("Model type:", type(model).__name__)

if hasattr(model, "medusa_head"):
    print("Number of Medusa heads:", len(model.medusa_head))

allocated = torch.cuda.memory_allocated() / 1024**3
peak = torch.cuda.max_memory_allocated() / 1024**3

print(f"Current GPU tensor memory: {allocated:.2f} GiB")
print(f"Peak GPU tensor memory: {peak:.2f} GiB")
print("Model loading check completed")
"""

process = subprocess.Popen(
    ["/content/medusa-env/bin/python", "-u", "-c", load_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    print(f"\nModel loading failed. Exit code: {return_code}")

/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available` instead.
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available`

In [10]:
import subprocess

load_code = r"""
import zipfile
import psutil
import torch
import transformers.modeling_utils as modeling_utils

from transformers import BitsAndBytesConfig
from medusa.model.medusa_model import MedusaModel

# Use memory mapping for PyTorch checkpoint shards.
original_load_state_dict = modeling_utils.load_state_dict

def load_state_dict_mmap(checkpoint_file):
    if str(checkpoint_file).endswith(".bin"):
        if not zipfile.is_zipfile(checkpoint_file):
            raise RuntimeError(
                "This checkpoint format does not support memory mapping."
            )

        return torch.load(
            checkpoint_file,
            map_location="cpu",
            mmap=True,
        )

    return original_load_state_dict(checkpoint_file)

modeling_utils.load_state_dict = load_state_dict_mmap

ram = psutil.virtual_memory()
print(f"Total system RAM: {ram.total / 1024**3:.2f} GiB")
print(f"Available system RAM: {ram.available / 1024**3:.2f} GiB")

model_id = "FasterDecoding/medusa-vicuna-7b-v1.3"

# Quantize the backbone while keeping prediction heads in FP16.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["lm_head", "medusa_head"],
)

print("Loading model with memory-mapped checkpoints...", flush=True)

try:
    model = MedusaModel.from_pretrained(
        model_id,
        quantization_config=quant_config,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        offload_state_dict=True,
        device_map={"": 0},
    )
finally:
    modeling_utils.load_state_dict = original_load_state_dict

model.eval()

print("\nModel loaded successfully")
print("Model type:", type(model).__name__)
print("Number of Medusa heads:", len(model.medusa_head))

allocated = torch.cuda.memory_allocated() / 1024**3
peak = torch.cuda.max_memory_allocated() / 1024**3

print(f"Current GPU tensor memory: {allocated:.2f} GiB")
print(f"Peak GPU tensor memory: {peak:.2f} GiB")
print("Model loading check completed")
"""

process = subprocess.Popen(
    ["/content/medusa-env/bin/python", "-u", "-c", load_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code == -9:
    print("\nThe process was killed by SIGKILL.")
    print("Check Colab system RAM usage; an out-of-memory kill is possible.")
elif return_code != 0:
    print(f"\nModel loading failed. Exit code: {return_code}")

/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available` instead.
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available`

In [11]:
import subprocess

test_code = r"""
from huggingface_hub import hf_hub_download
from fastchat.conversation import get_conv_template
import medusa.model.medusa_model as medusa_module
from medusa.model.utils import reset_medusa_mode

# Verify that the trained Medusa head weights were loaded.
head_path = hf_hub_download(model_id, "medusa_lm_head.pt")
saved_heads = torch.load(
    head_path,
    map_location="cpu",
    weights_only=True,
)
loaded_heads = model.medusa_head.state_dict()

assert saved_heads.keys() == loaded_heads.keys(), "Head parameter keys differ"

for name, parameter in loaded_heads.items():
    expected = saved_heads[name].to(
        device=parameter.device,
        dtype=parameter.dtype,
    )
    assert torch.equal(parameter, expected), f"Head weight mismatch: {name}"

del saved_heads, loaded_heads, expected
print("\nMedusa head weights verified")

# Prepare the same input for both decoding methods.
tokenizer = model.get_tokenizer()
question = "Explain what a robot is in one sentence."

conversation = get_conv_template("vicuna_v1.1")
conversation.append_message(conversation.roles[0], question)
conversation.append_message(conversation.roles[1], None)

inputs = tokenizer(
    conversation.get_prompt(),
    return_tensors="pt",
).to("cuda:0")

input_length = inputs.input_ids.shape[1]
max_new_tokens = 32

# Run ordinary greedy decoding as a functional reference.
# Caching is disabled here; this is not a speed benchmark.
print("Running greedy decoding...", flush=True)

with torch.inference_mode():
    greedy_output = model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=max_new_tokens,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

greedy_tokens = greedy_output[0, input_length:].tolist()

# Record accepted token IDs from the official Medusa decoding loop.
medusa_tokens = []
original_update = medusa_module.update_inference_inputs

def record_accepted_tokens(*args, **kwargs):
    result = original_update(*args, **kwargs)
    medusa_tokens[:] = result[0][0, input_length:].tolist()
    return result

medusa_module.update_inference_inputs = record_accepted_tokens
stream = None

print("Running Medusa greedy decoding...", flush=True)

try:
    with torch.inference_mode():
        stream = model.medusa_generate(
            inputs.input_ids,
            temperature=0.0,
            max_steps=max_new_tokens,
        )

        for _ in stream:
            if (
                len(medusa_tokens) >= max_new_tokens
                or tokenizer.eos_token_id in medusa_tokens
            ):
                break
finally:
    if stream is not None:
        stream.close()
    medusa_module.update_inference_inputs = original_update
    reset_medusa_mode(model)

# Apply the same output length and end-of-sequence limit.
medusa_tokens = medusa_tokens[:max_new_tokens]

if tokenizer.eos_token_id in medusa_tokens:
    end = medusa_tokens.index(tokenizer.eos_token_id) + 1
    medusa_tokens = medusa_tokens[:end]

print("\nQuestion:", question)
print("\nGreedy output:")
print(tokenizer.decode(greedy_tokens, skip_special_tokens=True))

print("\nMedusa output:")
print(tokenizer.decode(medusa_tokens, skip_special_tokens=True))

print("\nGreedy token count:", len(greedy_tokens))
print("Medusa token count:", len(medusa_tokens))
print("Exact token match:", greedy_tokens == medusa_tokens)
"""

process = subprocess.Popen(
    ["/content/medusa-env/bin/python", "-u", "-c", load_code + test_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    print(f"\nGeneration test failed. Exit code: {return_code}")

/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available` instead.
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available`

In [12]:
import subprocess

print("CELL 9: ACCEPTANCE TRACE", flush=True)

cell9_code = load_code + r"""
import medusa.model.medusa_model as medusa_module
from fastchat.conversation import get_conv_template
from medusa.model.utils import reset_medusa_mode

print("\nACCEPTANCE TRACE START", flush=True)

tokenizer = model.get_tokenizer()

conversation = get_conv_template("vicuna_v1.1")
conversation.append_message(
    conversation.roles[0],
    "Explain what a robot is in one sentence.",
)
conversation.append_message(conversation.roles[1], None)

input_ids = tokenizer(
    conversation.get_prompt(),
    return_tensors="pt",
).input_ids.to("cuda:0")

generated_tokens = []
round_lengths = []
max_new_tokens = 64
original_update = medusa_module.update_inference_inputs

def record_round(*args, **kwargs):
    previous_length = args[0].shape[1]
    result = original_update(*args, **kwargs)

    tokens = result[0][0, previous_length:].tolist()
    tokens = tokens[:max_new_tokens - len(generated_tokens)]

    if tokenizer.eos_token_id in tokens:
        tokens = tokens[:tokens.index(tokenizer.eos_token_id) + 1]

    generated_tokens.extend(tokens)
    round_lengths.append(len(tokens))

    text = tokenizer.decode(tokens, skip_special_tokens=False)
    print(
        f"{len(round_lengths):>5} | {len(tokens):>10} | {text!r}",
        flush=True,
    )
    return result

medusa_module.update_inference_inputs = record_round
stream = None

print("Round | Used tokens | Generated text", flush=True)
print("-" * 65, flush=True)

try:
    with torch.inference_mode():
        stream = model.medusa_generate(
            input_ids,
            temperature=0.0,
            max_steps=max_new_tokens,
        )

        for _ in stream:
            if (
                len(generated_tokens) >= max_new_tokens
                or tokenizer.eos_token_id in generated_tokens
            ):
                break
finally:
    if stream is not None:
        stream.close()
    medusa_module.update_inference_inputs = original_update
    reset_medusa_mode(model)

print("\nTokens accepted per round:", round_lengths)
print("Verification rounds:", len(round_lengths))
print("Total generated tokens:", len(generated_tokens))

if round_lengths:
    mean = len(generated_tokens) / len(round_lengths)
    print(f"Mean used tokens per verification: {mean:.2f}")

print("ACCEPTANCE TRACE END")
"""

cell9_process = subprocess.Popen(
    ["/content/medusa-env/bin/python", "-u", "-c", cell9_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in cell9_process.stdout:
    print(line, end="", flush=True)

exit_code = cell9_process.wait()

if exit_code != 0:
    print(f"\nAcceptance trace failed. Exit code: {exit_code}")

CELL 9: ACCEPTANCE TRACE
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available` instead.
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `i

In [13]:
import subprocess

batch_code = r"""
import medusa.model.medusa_model as medusa_module
from fastchat.conversation import get_conv_template
from medusa.model.utils import reset_medusa_mode

tokenizer = model.get_tokenizer()
max_new_tokens = 64

questions = [
    "Explain what a robot is in one sentence.",
    "What is the difference between weather and climate?",
    "Explain why the sky appears blue.",
    "Translate 'Good morning, how are you?' into French.",
    "Write a Python function that returns the square of a number.",
]

results = []
original_update = medusa_module.update_inference_inputs

for question_id, question in enumerate(questions, start=1):
    conversation = get_conv_template("vicuna_v1.1")
    conversation.append_message(conversation.roles[0], question)
    conversation.append_message(conversation.roles[1], None)

    inputs = tokenizer(
        conversation.get_prompt(),
        return_tensors="pt",
    ).to("cuda:0")

    input_length = inputs.input_ids.shape[1]
    reset_medusa_mode(model)

    # Obtain a greedy reference; this is not a speed benchmark.
    with torch.inference_mode():
        reference = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    reference_tokens = reference[0, input_length:].tolist()
    generated_tokens = []
    round_lengths = []

    # Record the effective output length of each verification round.
    def record_round(*args, **kwargs):
        previous_length = args[0].shape[1]
        result = original_update(*args, **kwargs)

        tokens = result[0][0, previous_length:].tolist()
        tokens = tokens[:max_new_tokens - len(generated_tokens)]

        if tokenizer.eos_token_id in tokens:
            tokens = tokens[:tokens.index(tokenizer.eos_token_id) + 1]

        generated_tokens.extend(tokens)
        round_lengths.append(len(tokens))
        return result

    medusa_module.update_inference_inputs = record_round
    stream = None

    try:
        with torch.inference_mode():
            stream = model.medusa_generate(
                inputs.input_ids,
                temperature=0.0,
                max_steps=max_new_tokens,
            )

            for _ in stream:
                if (
                    len(generated_tokens) >= max_new_tokens
                    or tokenizer.eos_token_id in generated_tokens
                ):
                    break
    finally:
        if stream is not None:
            stream.close()
        medusa_module.update_inference_inputs = original_update
        reset_medusa_mode(model)

    token_count = len(generated_tokens)
    round_count = len(round_lengths)
    matched = generated_tokens == reference_tokens

    results.append({
        "tokens": token_count,
        "rounds": round_count,
        "match": matched,
    })

    print(f"\nQuestion {question_id}: {question}", flush=True)
    print("Output:", tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ))
    print("Tokens accepted per round:", round_lengths)
    print("Exact token match:", matched)
    print(
        f"Tokens: {token_count} | Rounds: {round_count} | "
        f"Tokens per verification: {token_count / round_count:.2f}",
        flush=True,
    )

total_tokens = sum(row["tokens"] for row in results)
total_rounds = sum(row["rounds"] for row in results)
match_count = sum(row["match"] for row in results)

print("\nBATCH SUMMARY")
print("Questions tested:", len(results))
print(f"Exact token matches: {match_count}/{len(results)}")
print("Total generated tokens:", total_tokens)
print("Total verification rounds:", total_rounds)
print(
    "Overall tokens per verification:",
    f"{total_tokens / total_rounds:.2f}",
)
"""

process = subprocess.Popen(
    ["/content/medusa-env/bin/python", "-u", "-c", load_code + batch_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in process.stdout:
    print(line, end="", flush=True)

exit_code = process.wait()

if exit_code != 0:
    print(f"\nBatch check failed. Exit code: {exit_code}")

/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available` instead.
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available`

In [14]:
import json
import random
import subprocess
import urllib.request

from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount("/content/drive")

# Create a separate directory for this experiment.
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
project_dir = (
    Path("/content/drive/MyDrive/ECE285_Medusa")
    / run_id
)
project_dir.mkdir(parents=True, exist_ok=True)

# Pin the dataset revision for reproducibility.
dataset_id = "tatsu-lab/alpaca_eval"

with urllib.request.urlopen(
    f"https://huggingface.co/api/datasets/{dataset_id}",
    timeout=60,
) as response:
    revision = json.load(response)["sha"]

dataset_url = (
    f"https://huggingface.co/datasets/{dataset_id}"
    f"/resolve/{revision}/alpaca_eval.json"
)

with urllib.request.urlopen(dataset_url, timeout=60) as response:
    dataset = json.load(response)

assert isinstance(dataset, list)
assert len(dataset) >= 250
assert all(
    isinstance(row.get("instruction"), str)
    and row["instruction"].strip()
    for row in dataset
)

# Split by prompt so that training and validation remain separate.
seed = 42
indices = list(range(len(dataset)))
random.Random(seed).shuffle(indices)

train_indices = indices[:200]
validation_indices = indices[200:250]

def make_prompts(selected_indices):
    return [
        {
            "prompt_id": index,
            "instruction": dataset[index]["instruction"],
        }
        for index in selected_indices
    ]

train_prompts = make_prompts(train_indices)
validation_prompts = make_prompts(validation_indices)
pilot_prompts = train_prompts[:20]

assert set(train_indices).isdisjoint(validation_indices)

files = {
    "alpaca_eval_source.json": dataset,
    "train_prompts.json": train_prompts,
    "validation_prompts.json": validation_prompts,
    "pilot_prompts.json": pilot_prompts,
    "data_manifest.json": {
        "dataset": dataset_id,
        "revision": revision,
        "seed": seed,
        "train_prompt_ids": train_indices,
        "validation_prompt_ids": validation_indices,
        "pilot_prompt_ids": [
            row["prompt_id"] for row in pilot_prompts
        ],
    },
}

for filename, content in files.items():
    (project_dir / filename).write_text(
        json.dumps(content, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

# Save the working loader and an environment snapshot.
(project_dir / "load_model.py").write_text(
    load_code,
    encoding="utf-8",
)

environment = subprocess.run(
    [
        "uv", "pip", "freeze",
        "--python", "/content/medusa-env/bin/python",
    ],
    capture_output=True,
    text=True,
    check=True,
)

(project_dir / "environment_snapshot.txt").write_text(
    environment.stdout,
    encoding="utf-8",
)

print("\nDATA PREPARATION COMPLETE")
print("Dataset size:", len(dataset))
print("Training prompts:", len(train_prompts))
print("Validation prompts:", len(validation_prompts))
print("Pilot prompts:", len(pilot_prompts))
print("Train/validation overlap: 0")
print("Saved directory:", project_dir)
print("\nFirst pilot instruction:")
print(pilot_prompts[0]["instruction"])

Mounted at /content/drive

DATA PREPARATION COMPLETE
Dataset size: 805
Training prompts: 200
Validation prompts: 50
Pilot prompts: 20
Train/validation overlap: 0
Saved directory: /content/drive/MyDrive/ECE285_Medusa/20261028_192748_UTC

First pilot instruction:
What factors would you consider when designing an inclusive and accessible public transportation system?


In [16]:
import subprocess
from pathlib import Path
from google.colab import files

project_dir = Path(
    "/content/drive/MyDrive/ECE285_Medusa/20261028_192748_UTC"
)

script_path = project_dir / "collect_medusa_pilot.py"
N_PROMPTS = 20

# Upload the collector once and save it with the project.
if not script_path.exists():
    print("Upload collect_medusa_pilot.py")
    uploaded = files.upload()

    assert len(uploaded) == 1, "Please upload exactly one Python file."
    filename = next(iter(uploaded))
    assert filename.endswith(".py"), "Please upload a Python file."

    script_path.write_bytes(uploaded[filename])

print(f"Collecting candidate records for {N_PROMPTS} prompt(s)...")

process = subprocess.Popen(
    [
        "/content/medusa-env/bin/python",
        "-u",
        str(script_path),
        "--project-dir",
        str(project_dir),
        "--num-prompts",
        str(N_PROMPTS),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in process.stdout:
    print(line, end="", flush=True)

exit_code = process.wait()

if exit_code != 0:
    print(f"\nCollection failed. Exit code: {exit_code}")

/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available` instead.
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available`

In [17]:
import subprocess

diagnostic_code = r"""
import json
from pathlib import Path
from transformers import AutoTokenizer

record_path = Path(
    "/content/drive/MyDrive/ECE285_Medusa/20261028_192748_UTC"
    "/pilot_node_traces_v1/prompt_24.json"
)

record = json.loads(record_path.read_text(encoding="utf-8"))

tokenizer = AutoTokenizer.from_pretrained(
    "lmsys/vicuna-7b-v1.3",
    use_fast=False,
    local_files_only=True,
)

reference = record["reference_token_ids"]
generated = record["generated_token_ids"]

# Find the first unequal token or a difference in sequence length.
common_length = min(len(reference), len(generated))
position = next(
    (
        i for i in range(common_length)
        if reference[i] != generated[i]
    ),
    common_length,
)

print("\nMISMATCH DIAGNOSTIC")
print("Prompt ID:", record["prompt_id"])
print("Instruction:", record["instruction"])
print("Reference token count:", len(reference))
print("Medusa token count:", len(generated))
print("Matching prefix length:", position)

if reference == generated:
    print("The saved sequences match.")
else:
    print("First differing position, one-based:", position + 1)

    for name, tokens in [
        ("Reference", reference),
        ("Medusa", generated),
    ]:
        if position < len(tokens):
            token_id = tokens[position]
            print(f"{name} token ID:", token_id)
            print(
                f"{name} token text:",
                repr(tokenizer.decode([token_id])),
            )
        else:
            print(f"{name} sequence ends at this position.")

    print("\nShared prefix:")
    print(repr(tokenizer.decode(reference[:position])))

    print("\nReference continuation:")
    print(repr(tokenizer.decode(reference[position:position + 16])))

    print("\nMedusa continuation:")
    print(repr(tokenizer.decode(generated[position:position + 16])))

    consumed = 0
    for row in record["rounds"]:
        consumed += row["used_tokens"]
        if position < consumed:
            print("\nMedusa round containing the difference:", row["round"])
            break
"""

result = subprocess.run(
    ["/content/medusa-env/bin/python", "-u", "-c", diagnostic_code],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)


MISMATCH DIAGNOSTIC
Prompt ID: 24
Instruction: What breed dog is smallest?
Reference token count: 64
Medusa token count: 64
Matching prefix length: 31
First differing position, one-based: 32
Reference token ID: 29906
Reference token text: '2'
Medusa token ID: 29896
Medusa token text: '1'

Shared prefix:
'The smallest breed dog is the Chihuahua. Chihuahuas are known for their small size, typically weighing between '

Reference continuation:
'2-6 pounds (1-2.7 kg) and measuring around'

Medusa continuation:
'1-3 kg (2-7 lbs) and standing at around '

Medusa round containing the difference: 18



In [20]:
import subprocess
from pathlib import Path
from google.colab import files

project_dir = Path(
    "/content/drive/MyDrive/ECE285_Medusa/20261028_192748_UTC"
)
diagnostic_path = project_dir / "diagnose_medusa_mismatch.py"

if not diagnostic_path.exists():
    print("Upload diagnose_medusa_mismatch.py")
    uploaded = files.upload()

    assert len(uploaded) == 1, "Please upload exactly one Python file."
    filename = next(iter(uploaded))
    assert filename.endswith(".py"), "Please upload a Python file."

    diagnostic_path.write_bytes(uploaded[filename])

process = subprocess.Popen(
    [
        "/content/medusa-env/bin/python",
        "-u",
        str(diagnostic_path),
        "--project-dir",
        str(project_dir),
        "--prompt-id",
        "24",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in process.stdout:
    print(line, end="", flush=True)

exit_code = process.wait()

if exit_code != 0:
    print(f"\nMismatch diagnosis failed. Exit code: {exit_code}")

/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available` instead.
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available`